# Experimental test 6

* test difference btw Self_Consistency and Self_Consistency_re(refactored ver)
* for Self_Consistency_re  
    test ('vl',              # llm_model  
        3,                # few_shot_n  
        5,                # test_n(# of question for test)  
        'Y',              # q_src_yn  
        5,                # iteration num  
        'sys_prompt10',   # prompt ver  
        3,                # self-consistency number  
        0.01,             # temperature  
        'ver7'            # excel_verion  
        )  
* for Self_Consistency  
        test ('vl',              # llm_model  
            3,                # few_shot_n  
            5,                # test_n(# of question for test)  
            'Y',              # q_src_yn   
            6,                # iteration num  
            'sys_prompt10',   # prompt ver  
            3,                # self-consistency number  
            0.01,             # temperature  
            'ver7'            # excel_verion  
            )  
* compare the accuracy score   


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [2]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [3]:
def sc_calc_acc_condition_with_temp_with_sc_model(llm_model, model_ver, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}/{model_ver}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [4]:
    # test ('vl',              # llm_model
    #     3,                # few_shot_n
    #     5,                # test_n(# of question for test)
    #     'Y',              # q_src_yn 
    #     5,                # iteration num
    #     'sys_prompt10',   # prompt ver
    #     3,                # self-consistency number
    #     0.01,             # temperature
    #     'ver7'            # excel_verion
    #     )

In [5]:
# sc_vl_result_3_60_Y_100_sys_prompt10_5_0.01_ver7_99.csv
# stop        = ["</Difficulty Level>"] 있는 버전의 점수 
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 5, 'Y', 5, 'sys_prompt10', 3,  0.01, 'ver7')
print(list_)

len of opt_file : 5
              precision    recall  f1-score   support

           0      1.000     0.875     0.933         8
           1      0.857     0.857     0.857         7
           2      0.667     1.000     0.800         2

    accuracy                          0.882        17
   macro avg      0.841     0.911     0.863        17
weighted avg      0.902     0.882     0.886        17

vl_result_3_5_Y :  88.23529411764706
[np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(66.66666666666666), np.float64(75.0)]


In [6]:
        # test ('vl',              # llm_model
        #     3,                # few_shot_n
        #     5,                # test_n(# of question for test)
        #     'Y',              # q_src_yn 
        #     6,                # iteration num
        #     'sys_prompt10',   # prompt ver
        #     3,                # self-consistency number
        #     0.01,             # temperature
        #     'ver7'            # excel_verion
        #     )

In [7]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 5, 'Y', 6, 'sys_prompt10', 3,  0.01, 'ver7')

len of opt_file : 6
              precision    recall  f1-score   support

           0      1.000     0.818     0.900        11
           1      0.667     0.800     0.727         5
           2      0.500     1.000     0.667         1

    accuracy                          0.824        17
   macro avg      0.722     0.873     0.765        17
weighted avg      0.873     0.824     0.835        17

vl_result_3_5_Y :  82.35294117647058


In [8]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 100, 'Y', 50, 'sys_prompt10', 5,  0.01, 'ver7')
# print(list_)

len of opt_file : 50
              precision    recall  f1-score   support

           0      0.983     0.795     0.879      1390
           1      0.682     0.910     0.780       722
           2      0.884     0.944     0.913       373

    accuracy                          0.851      2485
   macro avg      0.850     0.883     0.857      2485
weighted avg      0.881     0.851     0.855      2485

vl_result_3_100_Y :  85.07042253521126


In [9]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 3, 50, 'Y', 50, 'sys_prompt10', 5,  0.01, 'ver7')
# print(list_)

len of opt_file : 50
              precision    recall  f1-score   support

           0      0.993     0.811     0.893       825
           1      0.775     0.901     0.833       679
           2      0.729     0.884     0.799       189

    accuracy                          0.855      1693
   macro avg      0.832     0.865     0.842      1693
weighted avg      0.876     0.855     0.858      1693

vq_result_3_50_Y :  85.52864737152983


In [10]:
list_ =         sc_calc_acc_condition_with_temp_with_sc_model('vq', 'models--Qwen--Qwen3-14B-AWQ',  3, 50, 'Y', 50, 'sys_prompt10', 5,  0.01, 'ver7')
# print(list_)

len of opt_file : 1


/tmp/ipykernel_801737/903763629.py:33: RuntimeWarning: invalid value encountered in scalar divide
  acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100


ValueError: max() arg is an empty sequence